#Transformar Dados de Drivers

- 1 - Ler a tabela drivers da camada bronze
- 2 - Manter apenas as colunas necessárias para análise (remover a coluna url)
- 3 - Padronizar os nomes das colunas usando snake_case (driverId → driver_id, date0fbirth → date_of_birth)
- 4 - Concatenar name.givenName e name.familyName para criar uma nova coluna chamada driver_name e transformar o valor para Title Case.
- 5 - Remover registros duplicados
- 6 - Transformar os valores das colunas nationality para Title Case
- 7 - Escrever os dados transformados na tabela constructors da camada silver 

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
from pyspark.sql import functions as F

1 - Ler a tabela drivers da camada bronze

In [0]:
drivers_df = spark.table(bronze_table)

2 - Manter apenas as colunas necessárias para análise (remover a coluna url)

In [0]:
drivers_drop_df = drivers_df.drop(F.col("url"))

3 - Padronizar os nomes das colunas usando snake_case (driverId → driver_id, date0fbirth → date_of_birth)

In [0]:
drivers_renamed_df = drivers_drop_df.withColumnsRenamed({
    "driverId": "driver_id",
    "dateOfBirth" : "date_of_birth"
})

In [0]:
display(drivers_renamed_df.select("name"))

name
"List(carlo, abate)"
"List(george, abecassis)"
"List(kenny, acheson)"
"List(philippe, adams)"
"List(walt, ader)"
"List(kurt, adolff)"
"List(fred, agabashian)"
"List(kurt, ahrens)"
"List(jack, aitken)"
"List(christijan, albers)"


4 - Concatenar name.givenName e name.familyName para criar uma nova coluna chamada driver_name e transformar o valor para Title Case.

In [0]:
 drivers_concatenated_df = (
     drivers_renamed_df
     .withColumn("driver_name", 
                 F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))))
     .drop("name")
 )

In [0]:
display(drivers_concatenated_df)

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,driver_name
abate,1932-07-10,italian,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Carlo Abate
abecassis,1913-03-21,british,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,George Abecassis
acheson,1957-11-27,british,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Kenny Acheson
adams,1969-11-19,belgian,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Philippe Adams
ader,1913-12-15,american,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Walt Ader
adolff,1921-11-05,german,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Kurt Adolff
agabashian,1913-08-21,american,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Fred Agabashian
ahrens,1940-04-19,german,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Kurt Ahrens
aitken,1995-09-23,british,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Jack Aitken
albers,1979-04-16,dutch,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Christijan Albers


In [0]:
drivers_duplicates_df = drivers_concatenated_df.dropDuplicates(["driver_id"])


In [0]:
drivers_final_df = ( drivers_duplicates_df
                    .withColumn('nationality', F.initcap(F.col("nationality")))
                    )

In [0]:
(
    drivers_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,driver_name
ahrens,1940-04-19,German,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Kurt Ahrens
barilla,1961-04-20,Italian,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Paolo Barilla
bayol,1914-02-28,French,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Élie Bayol
birger,1924-01-07,Argentine,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Pablo Birger
borgudd,1946-11-25,Swedish,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Slim Borgudd
branca,1916-09-15,Swiss,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Toni Branca
brown,1949-12-24,Australian,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Warwick Brown
christie,1924-04-04,American,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Bob Christie
clark,1936-03-04,British,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,Jim Clark
george_connor,1906-08-16,American,2026-08-21T19:01:30.714Z,dbfs:/Volumes/formula1/landing/arquivos/drivers.json,George Connor
